In [1]:
# pip install langchain_ollama

In [2]:
import os
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate


llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)
class Product(BaseModel):
        name:str = Field(description="Name of the product")
        price:str  = Field(description="Price of the product")
        description:str = Field(description="Description of the product")
        category:str = Field(description="Category of the product")
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", 
    temperature=0,
    api_key=os.environ["GEMINI_API_KEY"],
    ).with_structured_output(Product)

@tool
def product_detail(query: str) -> str: #"20+10"
    """Get the product detail from product"""
    pass

tools = [product_detail]



class AgentState(TypedDict):
    question: str
    decision: str
    tool_result: str
    answer: str


def agent(state: AgentState):

    question = state["question"]

    prompt = f"""
You are an AI Agent.

User Question:
{question}

Decide what to do.

If the question requires Product Details,
respond exactly:

USE_TOOL

Otherwise respond exactly:

DIRECT_ANSWER
"""
    response = llm.invoke(prompt)
    decision = response.content.strip()
    return {
        "decision": decision
    }
def use_tool(state: AgentState):
    data = open('data.txt').read()
    question = state["question"]

    template = '''
            You are a helpful assistant that can answer questions about the product.
            Answer the question based on the product details provided.
            product : {product}
            question : {question}
        '''
    prompt_template = PromptTemplate.from_template(template)
    chain = prompt_template | model
    result = chain.invoke({'product':data,'question':question})
    return {
        "tool_result": result
    }



def final_answer(state: AgentState):
    question = state["question"]
    tool_result = state.get("tool_result", "")

    if tool_result:

        prompt = f"""
Answer the user's question.

Question:
{question}

Product Detail:
{tool_result}

Give only the final answer.
"""

    else:

        prompt = f"""
Answer this question clearly:

{question}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }




def router(state: AgentState):

    decision = state["decision"]

    if "USE_TOOL" in decision:
        return "tool"

    return "answer"


graph = StateGraph(AgentState)

graph.add_node("agent", agent)
graph.add_node("tool", use_tool)
graph.add_node("answer", final_answer)

graph.add_edge(START, "agent")

graph.add_conditional_edges(
    "agent",
    router,
    {
        "tool": "tool",
        "answer": "answer"
    }
)

graph.add_edge("tool", "answer")
graph.add_edge("answer", END)




app = graph.compile()




result = app.invoke({
    "question": "Explain Machine Learning?",
    "decision": "",
    "tool_result": "",
    "answer": ""
})


print("\nAgent Answer:")
print(result)


Agent Answer:
{'question': 'Explain Machine Learning?', 'decision': 'Machine Learning is a subset of artificial intelligence that focuses on developing algorithms and statistical models that allow computer systems to learn and improve from experience without being explicitly programmed. It involves using data and statistical techniques to enable machines to recognize patterns, make predictions, and perform tasks that typically require human intelligence.\n\nKey aspects of Machine Learning include:\n\n1. Supervised Learning: Algorithms learn from labeled training data, where the correct answers are provided.\n2. Unsupervised Learning: Algorithms identify patterns in unlabeled data.\n3. Reinforcement Learning: Algorithms learn by interacting with an environment and receiving rewards or penalties.\n4. Deep Learning: A subset of Machine Learning that uses neural networks with many layers to learn complex representations of data.\n\nMachine Learning has numerous applications, including:\n\

In [3]:
print(result)

{'question': 'Explain Machine Learning?', 'decision': 'Machine Learning is a subset of artificial intelligence that focuses on developing algorithms and statistical models that allow computer systems to learn and improve from experience without being explicitly programmed. It involves using data and statistical techniques to enable machines to recognize patterns, make predictions, and perform tasks that typically require human intelligence.\n\nKey aspects of Machine Learning include:\n\n1. Supervised Learning: Algorithms learn from labeled training data, where the correct answers are provided.\n2. Unsupervised Learning: Algorithms identify patterns in unlabeled data.\n3. Reinforcement Learning: Algorithms learn by interacting with an environment and receiving rewards or penalties.\n4. Deep Learning: A subset of Machine Learning that uses neural networks with many layers to learn complex representations of data.\n\nMachine Learning has numerous applications, including:\n\n1. Image and s